# E1 · ejecución aislada de PetTrace v0.2.2

Ejecuta estas tres celdas en Colab o VS Code conectado a Colab. Solo necesitas un token de Hugging Face con lectura de `alessandro54/pettrace-reid` y `alessandro54/pettrace-training`, y credenciales de MLflow con permiso para registrar resultados en `pettrace-reid-eval`.

Este lanzador descarga la release versionada y crea **Python 3.11.15 en un entorno independiente**. El kernel de Colab solo inicia el proceso: sus paquetes NumPy/Pillow no participan en E1. No necesitas GPU ni reiniciar el kernel. Mantén la sesión conectada durante la ejecución.

Se evalúan los snapshots fijos de PetTrace v4 y el test público v1 por separado. La corrida solo está completa cuando ambos resultados y el notebook se publican en MLflow. Usa un directorio distinto para cada intento nuevo; las credenciales nunca se escriben en el notebook.


In [ ]:
import hashlib
import sys
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlopen
from uuid import uuid4

RELEASE = "0.2.2"
BASE = f"https://github.com/alessandro54/pettrace-ml/releases/download/v{RELEASE}"
with urlopen(f"{BASE}/SHA256SUMS") as response:
    inventory = {name: digest for digest, name in
                 (line.split("  ", 1) for line in response.read().decode().splitlines())}
with urlopen(f"{BASE}/run_e1.py") as response:
    script = response.read()
assert hashlib.sha256(script).hexdigest() == inventory["run_e1.py"], "Release checksum mismatch"
RUNNER = Path(f"run_e1-v{RELEASE}.py").resolve()
RUNNER.write_bytes(script)
WORK = Path("reports") / ("e1-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8])
WORK = WORK.resolve()
print("Release:", RELEASE, "| Evidence directory:", WORK)


In [ ]:
import getpass
import os
import warnings

try:
    from google.colab import userdata
except ImportError:
    userdata = None

os.environ["MLFLOW_TRACKING_URI"] = "https://ml.pettrace.app"
for name in ("HF_TOKEN", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    if not os.environ.get(name) and userdata is not None:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            pass
    if not os.environ.get(name, "").strip():
        with warnings.catch_warnings():
            warnings.simplefilter("error", getpass.GetPassWarning)
            os.environ[name] = getpass.getpass(f"{name}: ").strip()
    if not os.environ[name].strip():
        raise ValueError(f"Missing {name}")
print("Credentials loaded; values are not displayed.")


In [ ]:
import subprocess
from IPython.display import FileLink, display

subprocess.check_call([sys.executable, str(RUNNER), "--work-dir", str(WORK)])
display(FileLink(str(WORK / "e1.executed.ipynb")))
print((WORK / "publication.json").read_text())


## Si falla la publicación

Los reportes y `e1.executed.ipynb` permanecen en el directorio mostrado. Corrige las credenciales o la conexión y ejecuta en otra celda:

```python
subprocess.check_call([sys.executable, str(RUNNER), "--work-dir", str(WORK), "--resume"])
```

Esto reintenta únicamente la publicación de ambos dominios y adjunta la evidencia; no recalcula embeddings. Si el error ocurrió antes de terminar la evaluación, revisa el notebook guardado y comienza una corrida nueva después de resolverlo. Descarga la evidencia antes de descartar una sesión de Colab: su disco es temporal.
